<a href="https://colab.research.google.com/github/Agrannya-Singh/TuneTrace/blob/Version-2/YAMDA_RecSys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ploshkin_yambda_path = kagglehub.dataset_download('ploshkin/yambda')

print('Data source import complete.')


100%|██████████| 12.8G/12.8G [02:46<00:00, 82.6MB/s]

Extracting files...


Data source import complete.


In [4]:
"""
user_intent/sandbox_next_item_rec.py

Sandboxed Intent State Modeling & Sequential Next-Item Recommender.
Designed for the Samsung PRISM (SRI-B) Phase 2 Sandbox.

ETL & Modeling Stack:
1. Kaggle Local Parquet Search: Automatically scans '/kaggle/input' for local YAMDA parquets.
2. Hugging Face Datasets: Fallback streaming of YAMDA-50M parquets.
3. DuckDB: Executes high-speed joins, intent mapping, and sequence grouping directly on parquets.
4. SQLite: Writes the catalog, user logs, and model recommendations to 'recommendations.db' on disk.
5. FAISS: Indexes track vectors for HNSW similarity searches.
6. PyTorch (GRU) & Heuristic Profile Engine: Runs and compares both next-item prediction models.
"""

import os
import sqlite3
import logging
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict, Any, Optional

# Set up forced logging configuration (handles pre-initialized environments like Kaggle)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

# Double redundancy print logs for notebook output cells
def log_info(msg: str):
    logger.info(msg)
    print(f"[INFO] {msg}", flush=True)

def log_warn(msg: str):
    logger.warning(msg)
    print(f"[WARN] {msg}", flush=True)

def log_err(msg: str):
    logger.error(msg)
    print(f"[ERROR] {msg}", flush=True)

# Device Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_DIM = 256  # Default fallback dimension
SEQUENCE_LEN = 10     # User history sequence length
DECAY_LAMBDA = 1.0   # Decay lambda for heuristic profile engine

# File Path Configuration
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

DB_FILE_PATH = os.path.join(current_dir, "recommendations.db")

# =====================================================================
# Database Setup (SQLite)
# =====================================================================

class YambdaSandboxDatabase:
    """Manages SQLite relations for track catalogs, histories, and recommendations."""
    def __init__(self, db_path: str):
        # Remove old db if it exists to start fresh
        if os.path.exists(db_path) and db_path != ":memory:":
            try:
                os.remove(db_path)
            except Exception as e:
                log_warn(f"Could not remove old DB file: {e}")

        self.conn = sqlite3.connect(db_path)
        self.cursor = self.conn.cursor()
        self._create_tables()

    def _create_tables(self):
        # Active track catalog
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS tracks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                yamda_song_id INTEGER UNIQUE
            )
        ''')
        # User sequential logs with intent weights
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS user_histories (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER,
                track_id INTEGER,
                played_ratio_pct REAL,
                intent_state TEXT,
                intent_weight REAL,
                timestamp TEXT,
                FOREIGN KEY(track_id) REFERENCES tracks(id)
            )
        ''')
        # Combined model recommendations for comparison
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS recommendations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER,
                rank INTEGER,
                model_type TEXT, -- 'GRU_Seq' or 'Heuristic_IntentEngine'
                recommended_yamda_song_id INTEGER,
                score REAL
            )
        ''')
        self.conn.commit()

    def get_yamda_id(self, sqlite_id: int) -> int:
        self.cursor.execute("SELECT yamda_song_id FROM tracks WHERE id = ?", (sqlite_id,))
        row = self.cursor.fetchone()
        return row[0] if row else -1


# =====================================================================
# Local Parquet Search Helper
# =====================================================================

def locate_local_parquet(filename: str) -> Optional[str]:
    """Scans typical Kaggle and local directories to find parquets by filename pattern."""
    search_dirs = [
        "/kaggle/input",
        "./",
        "../",
    ]
    for base in search_dirs:
        if os.path.exists(base):
            for root, dirs, files in os.walk(base):
                for f in files:
                    if filename in f and f.endswith(".parquet"):
                        path = os.path.abspath(os.path.join(root, f))
                        log_info(f"Found local parquet file for '{filename}' at: {path}")
                        return path
    return None


# =====================================================================
# Real Data Ingestion (Kaggle Parquet / HF Streaming + DuckDB ETL)
# =====================================================================

def fetch_yamda_duckdb_data(
    db: YambdaSandboxDatabase,
    num_songs: int = 10000,
    num_listens: int = 150000
) -> Tuple[np.ndarray, Dict[int, List[Tuple[int, float, str, float]]], int]:
    """Loads YAMDA from local Kaggle parquets or falls back to Hugging Face streaming."""
    import duckdb

    local_embeddings = locate_local_parquet("embeddings.parquet")
    local_listens = locate_local_parquet("listens.parquet")

    # Flag to check if we can query parquets directly
    use_local_parquets = local_embeddings is not None and local_listens is not None

    if use_local_parquets:
        log_info("Local YAMDA parquet files detected. Verifying schemas dynamically...")
        con = duckdb.connect()

        # Dynamic Schema Resolution for Embeddings
        embed_column_names = [col[0] for col in con.execute(f"SELECT * FROM read_parquet('{local_embeddings}') LIMIT 0").description]
        log_info(f"Embeddings file columns: {embed_column_names}")

        vec_col = next((col for col in ["normalized_embed", "embed", "embedding", "features", "audio_embedding", "vector"] if col in embed_column_names), None)
        if not vec_col:
            raise ValueError(f"Could not find a valid embedding column in local parquet. Found: {embed_column_names}")

        embed_id_col = next((col for col in ["item_id", "item", "track_id"] if col in embed_column_names), "item_id")

        # Dynamic Schema Resolution for Listens
        listens_column_names = [col[0] for col in con.execute(f"SELECT * FROM read_parquet('{local_listens}') LIMIT 0").description]
        log_info(f"Listens file columns: {listens_column_names}")

        uid_col = next((col for col in ["uid", "user_id", "user"] if col in listens_column_names), "uid")
        listens_item_col = next((col for col in ["item_id", "item", "track_id"] if col in listens_column_names), "item_id")
        ratio_col = next((col for col in ["played_ratio_pct", "played_ratio", "ratio"] if col in listens_column_names), None)
        ts_col = next((col for col in ["timestamp", "ts", "time"] if col in listens_column_names), "timestamp")

        # Get column types for listens using robust DESCRIBE syntax
        listens_schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{local_listens}')").df()
        listens_types = dict(zip(listens_schema_df["column_name"], listens_schema_df["column_type"]))

        # Check if they are array types (DuckDB type string ends with '[]' or contains 'LIST')
        uid_is_array = "[]" in str(listens_types.get(uid_col, "")) or "LIST" in str(listens_types.get(uid_col, "")).upper()
        item_is_array = "[]" in str(listens_types.get(listens_item_col, "")) or "LIST" in str(listens_types.get(listens_item_col, "")).upper()
        ratio_is_array = ratio_col is not None and ("[]" in str(listens_types.get(ratio_col, "")) or "LIST" in str(listens_types.get(ratio_col, "")).upper())
        ts_is_array = "[]" in str(listens_types.get(ts_col, "")) or "LIST" in str(listens_types.get(ts_col, "")).upper()

        # Build SQL SELECT expressions to flatten the data
        uid_expr = f"UNNEST(l.{uid_col}) AS uid" if uid_is_array else f"l.{uid_col} AS uid"
        item_expr = f"UNNEST(l.{listens_item_col}) AS item_id" if item_is_array else f"l.{listens_item_col} AS item_id"

        if ratio_col:
            ratio_expr = f"UNNEST(l.{ratio_col}) AS played_ratio_pct" if ratio_is_array else f"l.{ratio_col} AS played_ratio_pct"
        else:
            ratio_expr = "100.0 AS played_ratio_pct"

        ts_expr = f"UNNEST(l.{ts_col}) AS timestamp" if ts_is_array else f"l.{ts_col} AS timestamp"

        log_info(f"Executing ETL on local parquets: uid={uid_col}, item={listens_item_col}, ratio={ratio_col}, vec={vec_col}...")

        # In local mode, we first flatten any sequential lists, then join with active track embeddings
        query_etl = f"""
            WITH unnested_listens AS (
                SELECT
                    {uid_expr},
                    {item_expr},
                    {ratio_expr},
                    {ts_expr}
                FROM read_parquet('{local_listens}') l
                LIMIT {num_listens}
            ),
            joined_data AS (
                SELECT
                    u.uid,
                    u.item_id,
                    CAST(u.played_ratio_pct AS REAL) AS played_ratio_pct,
                    u.timestamp,
                    CASE
                        WHEN u.played_ratio_pct >= 70.0 THEN 'PLAY_COMPLETE'
                        WHEN u.played_ratio_pct < 30.0 THEN 'SKIP'
                        ELSE 'PARTIAL_PLAY'
                    END AS intent_state,
                    CASE
                        WHEN u.played_ratio_pct >= 70.0 THEN 0.7
                        WHEN u.played_ratio_pct < 30.0 THEN -0.5
                        ELSE 0.3
                    END AS intent_weight
                FROM unnested_listens u
                INNER JOIN read_parquet('{local_embeddings}') e
                    ON u.item_id = e.{embed_id_col}
            ),
            user_counts AS (
                SELECT uid, count(*) as session_len
                FROM joined_data
                GROUP BY uid
                HAVING session_len > {SEQUENCE_LEN}
            )
            SELECT j.uid, j.item_id, j.played_ratio_pct, j.intent_state, j.intent_weight, j.timestamp
            FROM joined_data j
            INNER JOIN user_counts c ON j.uid = c.uid
            ORDER BY j.uid, j.timestamp
        """
        df_logs = con.execute(query_etl).df()

        if df_logs.empty:
            raise ValueError("Local ETL returned zero records. Check data filters.")

        # Extract active song catalog
        active_song_ids = df_logs["item_id"].unique().tolist()

        # Query active embeddings directly
        query_active_embeddings = f"""
            SELECT {embed_id_col} AS item_id, {vec_col} AS embedding
            FROM read_parquet('{local_embeddings}')
            WHERE {embed_id_col} IN (SELECT UNNEST(?))
            LIMIT {num_songs}
        """
        df_active_embeddings = con.execute(query_active_embeddings, [active_song_ids]).df()

    else:
        # Fallback to Hugging Face datasets streaming
        log_info("Local YAMDA parquets not found. Falling back to Hugging Face streaming loader...")
        try:
            from datasets import load_dataset
        except ImportError:
            raise ImportError(
                "Local parquets are missing and the 'datasets' package is not installed. "
                "Run: pip install datasets"
            )

        # 1. Stream CNN Embeddings
        embeddings_ds = load_dataset("yandex/yambda", data_files="embeddings.parquet", streaming=True)["train"]
        embeddings_chunk = list(embeddings_ds.take(num_songs))

        embeddings_list = []
        for row in embeddings_chunk:
            vec_col = next((col for col in ["normalized_embed", "embed", "embedding", "features", "audio_embedding", "vector"] if col in row), None)
            item_id = row.get("item_id", row.get("item"))
            if vec_col and row[vec_col] is not None and item_id is not None:
                embeddings_list.append(
                    {
                        "item_id": int(item_id),
                        "embedding": row[vec_col]
                    }
                )

        if not embeddings_list:
            raise ValueError("Could not extract any embeddings from Hugging Face stream.")

        df_raw_embeddings = pd.DataFrame(embeddings_list)

        # 2. Stream user logs
        listens_ds = load_dataset("yandex/yambda", data_dir="sequential/50m", data_files="listens.parquet", streaming=True)["train"]
        listens_chunk = list(listens_ds.take(num_listens))

        listens_list = []
        for row in listens_chunk:
            uid = row.get("uid", row.get("user_id"))
            item_id = row.get("item_id", row.get("item"))
            ratio = row.get("played_ratio_pct", 100.0)
            ts = row.get("timestamp", "")
            if uid is not None and item_id is not None:
                listens_list.append(
                    {
                        "uid": int(uid),
                        "item_id": int(item_id),
                        "played_ratio_pct": float(ratio) if ratio is not None else 100.0,
                        "timestamp": str(ts)
                    }
                )
        df_raw_listens = pd.DataFrame(listens_list)

        # 3. DuckDB processing
        con = duckdb.connect()
        query_etl = f"""
            WITH joined_data AS (
                SELECT
                    l.uid,
                    l.item_id,
                    l.played_ratio_pct,
                    l.timestamp,
                    CASE
                        WHEN l.played_ratio_pct >= 70.0 THEN 'PLAY_COMPLETE'
                        WHEN l.played_ratio_pct < 30.0 THEN 'SKIP'
                        ELSE 'PARTIAL_PLAY'
                    END AS intent_state,
                    CASE
                        WHEN l.played_ratio_pct >= 70.0 THEN 0.7
                        WHEN l.played_ratio_pct < 30.0 THEN -0.5
                        ELSE 0.3
                    END AS intent_weight
                FROM df_raw_listens l
                INNER JOIN df_raw_embeddings e
                    ON l.item_id = e.item_id
            ),
            user_counts AS (
                SELECT uid, count(*) as session_len
                FROM joined_data
                GROUP BY uid
                HAVING session_len > {SEQUENCE_LEN}
            )
            SELECT j.uid, j.item_id, j.played_ratio_pct, j.intent_state, j.intent_weight, j.timestamp
            FROM joined_data j
            INNER JOIN user_counts c ON j.uid = c.uid
            ORDER BY j.uid, j.timestamp
        """
        df_logs = con.execute(query_etl).df()

        if df_logs.empty:
            raise ValueError("ETL returned zero records.")

        active_song_ids = df_logs["item_id"].unique().tolist()

        query_active_embeddings = """
            SELECT item_id, embedding
            FROM df_raw_embeddings
            WHERE item_id IN (SELECT UNNEST(?))
        """
        df_active_embeddings = con.execute(query_active_embeddings, [active_song_ids]).df()

    # 4. Ingest filtered records into SQLite
    db.cursor.execute("BEGIN TRANSACTION;")
    for song_id in active_song_ids:
        db.cursor.execute("INSERT OR IGNORE INTO tracks (yamda_song_id) VALUES (?) ", (song_id,))
    db.cursor.execute("COMMIT;")

    db.cursor.execute("SELECT id, yamda_song_id FROM tracks")
    db_tracks = db.cursor.fetchall()
    yamda_to_sqlite = {row[1]: row[0] for row in db_tracks}

    db.cursor.execute("BEGIN TRANSACTION;")
    user_histories = {}
    for _, row in df_logs.iterrows():
        uid = int(row["uid"])
        sid = int(row["item_id"])
        if sid not in yamda_to_sqlite:
            continue
        sqlite_id = yamda_to_sqlite[sid]
        ratio = float(row["played_ratio_pct"])
        state = str(row["intent_state"])
        weight = float(row["intent_weight"])
        ts = str(row["timestamp"])

        db.cursor.execute(
            """INSERT INTO user_histories
               (user_id, track_id, played_ratio_pct, intent_state, intent_weight, timestamp)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (uid, sqlite_id, ratio, state, weight, ts)
        )

        if uid not in user_histories:
            user_histories[uid] = []
        user_histories[uid].append((sqlite_id, ratio, state, weight))
    db.cursor.execute("COMMIT;")

    # Align embedding matrix
    num_tracks_in_db = len(yamda_to_sqlite)
    dim = len(df_active_embeddings["embedding"].iloc[0])
    aligned_embeddings = np.zeros((num_tracks_in_db, dim), dtype=np.float32)

    for _, row in df_active_embeddings.iterrows():
        sid = int(row["item_id"])
        if sid in yamda_to_sqlite:
            sqlite_idx = yamda_to_sqlite[sid] - 1
            aligned_embeddings[sqlite_idx] = row["embedding"]

    # Normalize L2 (add epsilon to prevent divide-by-zero for missing embeddings)
    norms = np.linalg.norm(aligned_embeddings, axis=1, keepdims=True)
    aligned_embeddings /= (norms + 1e-9)

    log_info(
        f"DuckDB ETL Complete: Loaded {len(user_histories)} users and "
        f"{num_tracks_in_db} tracks into recommendations.db."
    )
    return aligned_embeddings, user_histories, dim


# =====================================================================
# Synthetic Fallback Ingestion
# =====================================================================

def populate_mock_data(
    db: YambdaSandboxDatabase,
    num_songs: int = 500,
    num_users: int = 50
) -> Tuple[np.ndarray, Dict[int, List[Tuple[int, float, str, float]]], int]:
    log_warn("Falling back to synthetic mock data generation...")

    yamda_song_ids = [200000 + i for i in range(num_songs)]
    db.cursor.execute("BEGIN TRANSACTION;")
    for song_id in yamda_song_ids:
        db.cursor.execute("INSERT OR IGNORE INTO tracks (yamda_song_id) VALUES (?) ", (song_id,))
    db.cursor.execute("COMMIT;")

    db.cursor.execute("SELECT id, yamda_song_id FROM tracks")
    db_tracks = db.cursor.fetchall()
    sqlite_track_ids = [row[0] for row in db_tracks]

    user_histories = {}
    db.cursor.execute("BEGIN TRANSACTION;")
    for user_id in range(num_users):
        seq_len = random.randint(8, 12)
        user_histories[user_id] = []
        for _ in range(seq_len):
            track_id = random.choice(sqlite_track_ids)
            ratio = random.choice([10.0, 50.0, 95.0])
            state = 'SKIP' if ratio < 30 else ('PLAY_COMPLETE' if ratio >= 70 else 'PARTIAL_PLAY')
            weight = -0.5 if state == 'SKIP' else (0.7 if state == 'PLAY_COMPLETE' else 0.3)

            db.cursor.execute(
                """INSERT INTO user_histories
                   (user_id, track_id, played_ratio_pct, intent_state, intent_weight, timestamp)
                   VALUES (?, ?, ?, ?, ?, datetime('now'))""",
                (user_id, track_id, ratio, state, weight)
            )
            user_histories[user_id].append((track_id, ratio, state, weight))
    db.cursor.execute("COMMIT;")

    embeddings = np.random.randn(num_songs, EMBEDDING_DIM).astype(np.float32)
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

    return embeddings, user_histories, EMBEDDING_DIM


# =====================================================================
# FAISS HNSW Index Setup
# =====================================================================

def build_faiss_hnsw_index(embeddings: np.ndarray, dim: int) -> Any:
    try:
        import faiss
    except ImportError:
        log_warn("FAISS is not installed. Attempting to install 'faiss-cpu' dynamically...")
        try:
            import subprocess
            import sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])
            import faiss
            log_info("Successfully installed FAISS in the current environment.")
        except Exception as e:
            log_warn(f"Failed to install FAISS: {e}. Defaulting to Python cosine index.")
            return PythonCosineIndex(embeddings)

    log_info("Building FAISS HNSW Index for ultra-fast ANN search...")
    normalized_embeds = embeddings.astype('float32').copy()
    faiss.normalize_L2(normalized_embeds)

    index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
    index.add(normalized_embeds)
    return index

class PythonCosineIndex:
    def __init__(self, embeddings: np.ndarray):
        self.embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
        self.ntotal = len(embeddings)

    def search(self, query: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        q_norm = query / np.linalg.norm(query, axis=1, keepdims=True)
        similarities = np.dot(self.embeddings, q_norm.T).T
        indices = np.argsort(-similarities, axis=1)[:, :k]
        distances = np.take_along_axis(similarities, indices, axis=1)
        return distances, indices


# =====================================================================
# PyTorch GRU Sequential Model & Dataset
# =====================================================================

class SequentialRecDataset(Dataset):
    def __init__(self, user_histories: Dict[int, List[Tuple[int, float, str, float]]], embeddings: np.ndarray, seq_len: int = SEQUENCE_LEN):
        self.sequences = []
        self.targets = []
        self.embeddings = embeddings

        for user_id, history in user_histories.items():
            if len(history) <= seq_len:
                continue
            for i in range(len(history) - seq_len):
                seq_ids = [item[0] - 1 for item in history[i : i + seq_len]]
                target_id = history[i + seq_len][0] - 1

                self.sequences.append(seq_ids)
                self.targets.append(target_id)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq_embeds = self.embeddings[self.sequences[idx]]
        target_embed = self.embeddings[self.targets[idx]]
        return torch.tensor(seq_embeds, dtype=torch.float32), torch.tensor(target_embed, dtype=torch.float32)


class GRUSeqRecommender(nn.Module):
    def __init__(self, embedding_dim: int, hidden_dim: int = 128, num_layers: int = 2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gru_out, _ = self.gru(x)
        last_step_out = gru_out[:, -1, :]
        out = self.fc(last_step_out)
        return nn.functional.normalize(out, p=2, dim=1)


# =====================================================================
# Heuristic Intent Profile Engine
# =====================================================================

class HeuristicIntentEngine:
    def __init__(self, decay_lambda: float = DECAY_LAMBDA):
        self.decay_lambda = decay_lambda

    def compute_heuristic_profile(
        self,
        history_slice: List[Tuple[int, float, str, float]],
        embeddings: np.ndarray
    ) -> np.ndarray:
        n = len(history_slice)
        if n == 0:
            return np.zeros(embeddings.shape[1], dtype=np.float32)

        slice_embeddings = np.array([embeddings[item[0] - 1] for item in history_slice], dtype=np.float32)
        signal_weights = np.array([item[3] for item in history_slice], dtype=np.float32)

        steps_from_recent = np.arange(n - 1, -1, -1, dtype=np.float32)
        decay = np.exp(-self.decay_lambda * steps_from_recent)

        combined_weights = signal_weights * decay
        raw_profile = (combined_weights[:, np.newaxis] * slice_embeddings).sum(axis=0)

        norm = np.linalg.norm(raw_profile)
        if norm < 1e-8:
            return np.zeros(embeddings.shape[1], dtype=np.float32)
        return raw_profile / norm


# =====================================================================
# Main Execution Pipeline
# =====================================================================

def run_sandbox_training():
    log_info("Initializing Large-Scale Sandbox Ingestion...")
    log_info(f"Target SQLite Database File: {DB_FILE_PATH}")
    db = YambdaSandboxDatabase(DB_FILE_PATH)

    # Ingestion Block using DuckDB
    try:
        raw_embeddings, user_histories, actual_dim = fetch_yamda_duckdb_data(
            db,
            num_songs=100000,
            num_listens=2000000
        )
        log_info("Successfully loaded real YAMDA dataset via DuckDB.")
    except Exception as e:
        log_err(f"Failed to load real data: {e}.")
        raw_embeddings, user_histories, actual_dim = populate_mock_data(db, num_songs=1000, num_users=200)

    if not user_histories:
        log_err("No active user sequences were parsed. Exiting.")
        return

    # Build HNSW Index
    hnsw_index = build_faiss_hnsw_index(raw_embeddings, actual_dim)

    # Setup Heuristic Engine
    heuristic_engine = HeuristicIntentEngine(decay_lambda=DECAY_LAMBDA)

    # Prepare PyTorch Dataloader
    dataset = SequentialRecDataset(user_histories, raw_embeddings, seq_len=SEQUENCE_LEN)
    batch_size = min(2048, len(dataset))
    if batch_size < 2:
        log_err("Dataset size is too small for PyTorch sequence batching.")
        return

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # Initialize PyTorch Sequence Model
    model = GRUSeqRecommender(embedding_dim=actual_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CosineEmbeddingLoss()
    target_similarity_labels = torch.ones(batch_size).to(DEVICE)

    # Train sequential model
    epochs = 10
    model.train()
    log_info(f"Training GRU Sequence model for {epochs} epochs on device: {DEVICE}...")
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (batch_seq, batch_target) in enumerate(loader):
            batch_seq = batch_seq.to(DEVICE)
            batch_target = batch_target.to(DEVICE)

            optimizer.zero_grad()
            pred_next_embedding = model(batch_seq)

            # Slice labels to handle drop_last or remaining batch size properly
            labels = target_similarity_labels[:batch_seq.size(0)]
            loss = criterion(pred_next_embedding, batch_target, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f"  [Epoch {epoch+1}] Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}", flush=True)

        avg_loss = total_loss / len(loader)
        log_info(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    # Generate Recommendations and Save to SQL
    model.eval()
    log_info("==================================================================")
    log_info(f"Generating side-by-side next-item recommendations for {len(user_histories)} users...")
    log_info("==================================================================")

    db.cursor.execute("BEGIN TRANSACTION;")

    for uid, history in user_histories.items():
        last_history_slice = history[-SEQUENCE_LEN:]
        last_seq_ids = [item[0] - 1 for item in last_history_slice]
        seq_embeds = raw_embeddings[last_seq_ids]

        # Model 1: GRU Sequence Model
        input_tensor = torch.tensor(seq_embeds, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            gru_predicted_vector = model(input_tensor).cpu().numpy()

        distances_gru, indices_gru = hnsw_index.search(gru_predicted_vector, k=5)
        for rank, (idx, dist) in enumerate(zip(indices_gru[0], distances_gru[0])):
            sqlite_id = int(idx) + 1
            yamda_song_id = db.get_yamda_id(sqlite_id)
            db.cursor.execute(
                """INSERT INTO recommendations (user_id, rank, model_type, recommended_yamda_song_id, score)
                   VALUES (?, ?, 'GRU_Seq', ?, ?)""",
                (uid, rank + 1, yamda_song_id, float(dist))
            )

        # Model 2: Heuristic IntentEngine (Decay Weighted Average)
        heuristic_profile = heuristic_engine.compute_heuristic_profile(last_history_slice, raw_embeddings)

        # Query FAISS
        distances_heur, indices_heur = hnsw_index.search(np.expand_dims(heuristic_profile, axis=0), k=5)
        for rank, (idx, dist) in enumerate(zip(indices_heur[0], distances_heur[0])):
            sqlite_id = int(idx) + 1
            yamda_song_id = db.get_yamda_id(sqlite_id)
            db.cursor.execute(
                """INSERT INTO recommendations (user_id, rank, model_type, recommended_yamda_song_id, score)
                   VALUES (?, ?, 'Heuristic_IntentEngine', ?, ?)""",
                (uid, rank + 1, yamda_song_id, float(dist))
            )

    db.conn.commit()
    log_info("✅ All recommendations successfully saved to recommendations.db.")

    # Print sample recommendations for the first 5 users
    test_users = list(user_histories.keys())[:5]
    for uid in test_users:
        log_info(f"User {uid} sequential history (last 5): {[db.get_yamda_id(item[0]) for item in user_histories[uid][-5:]]}")

        # Query GRU Recs
        db.cursor.execute(
            "SELECT recommended_yamda_song_id, score FROM recommendations WHERE user_id = ? AND model_type = 'GRU_Seq' ORDER BY rank",
            (uid,)
        )
        gru_recs = db.cursor.fetchall()
        log_info(f" ├─ GRU Recommender Top 5: {[row[0] for row in gru_recs]}")

        # Query Heuristic Recs
        db.cursor.execute(
            "SELECT recommended_yamda_song_id, score FROM recommendations WHERE user_id = ? AND model_type = 'Heuristic_IntentEngine' ORDER BY rank",
            (uid,)
        )
        heur_recs = db.cursor.fetchall()
        log_info(f" └─ Heuristic Engine Top 5: {[row[0] for row in heur_recs]}")
        log_info("------------------------------------------------------------------")

    return raw_embeddings, user_histories, hnsw_index, heuristic_engine, model


if __name__ == "__main__":
    # Assign the returned values to global variables
    global raw_embeddings, user_histories, hnsw_index, heuristic_engine, model
    raw_embeddings, user_histories, hnsw_index, heuristic_engine, model = run_sandbox_training()

2026-07-13 20:19:27,413 [INFO] Initializing Large-Scale Sandbox Ingestion...


[INFO] Initializing Large-Scale Sandbox Ingestion...


2026-07-13 20:19:27,414 [INFO] Target SQLite Database File: /content/recommendations.db


[INFO] Target SQLite Database File: /content/recommendations.db


2026-07-13 20:19:29,491 [INFO] Found local parquet file for 'embeddings.parquet' at: /root/.cache/kagglehub/datasets/ploshkin/yambda/versions/1/embeddings.parquet


...

[INFO]  ├─ GRU Recommender Top 5: [5274522, 335807, 4528740, 1521466, 5869779]


2026-07-13 20:34:34,393 [INFO]  └─ Heuristic Engine Top 5: [351321, 2953396, 439723, 2608372, 3378815]


[INFO]  └─ Heuristic Engine Top 5: [351321, 2953396, 439723, 2608372, 3378815]


2026-07-13 20:34:34,394 [INFO] ------------------------------------------------------------------


[INFO] ------------------------------------------------------------------


In [7]:
import sqlite3
import pandas as pd
import duckdb
import os
import glob

db_path = "recommendations.db"
conn = sqlite3.connect(db_path)

# 1. Pick a random user ID from the database
test_user_id = pd.read_sql_query("SELECT DISTINCT user_id FROM recommendations LIMIT 1 OFFSET 3;", conn).iloc[0, 0]

# 2. Get the user's history and recommendations (Track IDs)
history_df = pd.read_sql_query(f"""
    SELECT track_id AS yamda_id, intent_state, played_ratio_pct
    FROM user_histories WHERE user_id = {test_user_id}
    ORDER BY timestamp DESC LIMIT 5;
""", conn)

gru_df = pd.read_sql_query(f"""
    SELECT recommended_yamda_song_id AS yamda_id, rank, score
    FROM recommendations
    WHERE user_id = {test_user_id} AND model_type = 'GRU_Seq' ORDER BY rank ASC;
""", conn)

heur_df = pd.read_sql_query(f"""
    SELECT recommended_yamda_song_id AS yamda_id, rank, score
    FROM recommendations
    WHERE user_id = {test_user_id} AND model_type = 'Heuristic_IntentEngine' ORDER BY rank ASC;
""", conn)
conn.close()

# 3. Dynamically locate the YAMDA metadata/tracks file
base_dir = "/kaggle/input/datasets/ploshkin/yambda"
possible_files = glob.glob(f"{base_dir}/*meta*.parquet") + glob.glob(f"{base_dir}/*track*.parquet") + glob.glob(f"{base_dir}/*item*.parquet")

metadata_file = possible_files[0] if possible_files else None

def attach_metadata(df, title):
    print(f"\n{'='*50}\n{title}\n{'='*50}")
    if metadata_file and not df.empty:
        try:
            # Let DuckDB auto-detect the columns and join
            joined_df = duckdb.query(f"""
                SELECT d.*, t.* EXCLUDE (item_id)
                FROM df d
                LEFT JOIN read_parquet('{metadata_file}') t
                ON d.yamda_id = t.item_id
            """).df()
            display(joined_df)
            return
        except Exception as e:
            print(f"Join failed: {e}. Showing raw IDs instead.")

    display(df)

if metadata_file:
    print(f"✅ Found metadata file: {metadata_file}")
else:
    print("⚠️ Could not find any metadata/tracks parquet file in the Kaggle directory.")

attach_metadata(history_df, f"USER {test_user_id} - RECENT HISTORY")
attach_metadata(gru_df, f"USER {test_user_id} - GRU RECOMMENDATIONS")
attach_metadata(heur_df, f"USER {test_user_id} - HEURISTIC RECOMMENDATIONS")


⚠️ Could not find any metadata/tracks parquet file in the Kaggle directory.

USER 500 - RECENT HISTORY


,yamda_id,intent_state,played_ratio_pct
0,1463,SKIP,13.0
1,1462,SKIP,0.0
2,1461,PLAY_COMPLETE,100.0
3,1460,PLAY_COMPLETE,100.0
4,1459,PLAY_COMPLETE,100.0



USER 500 - GRU RECOMMENDATIONS


,yamda_id,rank,score
0,1286879,1,0.950791
1,3298994,2,0.942962
2,393082,3,0.942494
3,6059016,4,0.942123
4,4649904,5,0.942101



USER 500 - HEURISTIC RECOMMENDATIONS


,yamda_id,rank,score
0,389515,1,0.203907
1,508821,2,0.199597
2,5013179,3,0.191880
3,6176417,4,0.178974
4,2471852,5,0.164567


In [10]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('recommendations.db')

# 1. Grab 20 Random Users
users_df = pd.read_sql_query("SELECT DISTINCT user_id FROM recommendations LIMIT 20;", conn)
user_ids = users_df['user_id'].tolist()

sheet_data = []

for uid in user_ids:
    # Get last 5 Listened Tracks (History)
    history = pd.read_sql_query(f"""
        SELECT track_id FROM user_histories
        WHERE user_id = {uid} ORDER BY timestamp DESC LIMIT 5;
    """, conn)['track_id'].tolist()

    # Get Top-5 GRU Recommendations
    gru_recs = pd.read_sql_query(f"""
        SELECT recommended_yamda_song_id FROM recommendations
        WHERE user_id = {uid} AND model_type = 'GRU_Seq' ORDER BY rank ASC LIMIT 5;
    """, conn)['recommended_yamda_song_id'].tolist()

    # Get Top-5 Heuristic Recommendations
    heur_recs = pd.read_sql_query(f"""
        SELECT recommended_yamda_song_id FROM recommendations
        WHERE user_id = {uid} AND model_type = 'Heuristic_IntentEngine' ORDER BY rank ASC LIMIT 5;
    """, conn)['recommended_yamda_song_id'].tolist()

    # Pad lists with None if they have less than 5 items (safeguard)
    history += [None] * (5 - len(history))
    gru_recs += [None] * (5 - len(gru_recs))
    heur_recs += [None] * (5 - len(heur_recs))

    # Build the row dictionary
    row = {
        'User ID': uid,
        'History 1 (Most Recent)': history[0],
        'History 2': history[1],
        'History 3': history[2],
        'History 4': history[3],
        'History 5': history[4],
        'GRU Prediction 1': gru_recs[0],
        'GRU Prediction 2': gru_recs[1],
        'GRU Prediction 3': gru_recs[2],
        'GRU Prediction 4': gru_recs[3],
        'GRU Prediction 5': gru_recs[4],
        'Heuristic Prediction 1': heur_recs[0],
        'Heuristic Prediction 2': heur_recs[1],
        'Heuristic Prediction 3': heur_recs[2],
        'Heuristic Prediction 4': heur_recs[3],
        'Heuristic Prediction 5': heur_recs[4],
    }
    sheet_data.append(row)

conn.close()

# 2. Convert to DataFrame and Export to CSV
export_df = pd.DataFrame(sheet_data)
export_df.to_csv('recommendations_export.csv', index=False)

print(" Successfully generated 'recommendations_export.csv'!")
display(export_df.head())  # Show a quick preview in the notebook


 Successfully generated 'recommendations_export.csv'!


,User ID,History 1 (Most Recent),History 2,History 3,History 4,History 5,GRU Prediction 1,GRU Prediction 2,GRU Prediction 3,GRU Prediction 4,GRU Prediction 5,Heuristic Prediction 1,Heuristic Prediction 2,Heuristic Prediction 3,Heuristic Prediction 4,Heuristic Prediction 5
0,100,319,304,532,584,511,3443475,4531263,5678553,3781548,3537376,4734787,3426714,5150945,451318,2077046
1,200,967,902,1058,1059,1060,5811656,4219887,3361052,1252472,3778807,5134208,75595,3390913,1389443,4557096
2,300,1176,1175,1174,1173,1172,1664309,3376865,1152738,4422796,2248488,29578,3674609,5739503,2732146,5587903
3,500,1463,1462,1461,1460,1459,1286879,3298994,393082,6059016,4649904,389515,508821,5013179,6176417,2471852
4,600,1760,1759,1758,1757,1756,5274522,335807,4528740,1521466,5869779,351321,2953396,439723,2608372,3378815


In [13]:
import sqlite3
import pandas as pd
import numpy as np
import duckdb
import glob

print("Fetching history and recommendations directly from SQLite...")
conn = sqlite3.connect('recommendations.db')

# 1. Get the last 5 songs the user listened to (their recent 'vibe')
history_df = pd.read_sql_query("""
    SELECT user_id, track_id AS yamda_song_id
    FROM (
        SELECT user_id, track_id,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY timestamp DESC) as rn
        FROM user_histories
    ) WHERE rn <= 5
""", conn)

# 2. Get the saved recommendations
gru_df = pd.read_sql_query("SELECT user_id, recommended_yamda_song_id AS yamda_song_id FROM recommendations WHERE model_type = 'GRU_Seq'", conn)
heur_df = pd.read_sql_query("SELECT user_id, recommended_yamda_song_id AS yamda_song_id FROM recommendations WHERE model_type = 'Heuristic_IntentEngine'", conn)
conn.close()

# 3. Search the kagglehub cache directory for the downloaded embeddings file
print("Searching for the downloaded YAMDA dataset...")
search_paths = glob.glob("/root/.cache/kagglehub/datasets/ploshkin/yambda/**/embeddings.parquet", recursive=True)

if not search_paths:
    raise FileNotFoundError("Could not find embeddings.parquet in the kagglehub cache! Are you sure it finished extracting?")

embeddings_path = search_paths[0]
print(f"✅ Found local embeddings file at: {embeddings_path}")

con = duckdb.connect()

# 4. Use DuckDB to attach the 384-d vectors to our tracking dataframes
def get_embeds(df):
    return con.execute(f"""
        SELECT d.user_id, e.normalized_embed
        FROM df d
        JOIN read_parquet('{embeddings_path}') e ON d.yamda_song_id = e.item_id
    """).df()

print("Fetching vectors for fast comparison...")
hist_embeds = get_embeds(history_df)
gru_embeds = get_embeds(gru_df)
heur_embeds = get_embeds(heur_df)

print("Calculating Average Semantic Similarity Matrix...")
def calc_similarity(hist_vecs, rec_vecs):
    if len(hist_vecs) == 0 or len(rec_vecs) == 0: return 0.0
    h_matrix = np.stack(hist_vecs.values)
    r_matrix = np.stack(rec_vecs.values)
    sim_matrix = np.dot(h_matrix, r_matrix.T) # Calculate Cosine Similarity
    return np.mean(np.max(sim_matrix, axis=1))

gru_sims = []
heur_sims = []

for uid, h_grp in hist_embeds.groupby('user_id'):
    g_grp = gru_embeds[gru_embeds['user_id'] == uid]['normalized_embed']
    h_e_grp = heur_embeds[heur_embeds['user_id'] == uid]['normalized_embed']

    if len(g_grp) > 0 and len(h_e_grp) > 0:
        gru_sims.append(calc_similarity(h_grp['normalized_embed'], g_grp))
        heur_sims.append(calc_similarity(h_grp['normalized_embed'], h_e_grp))

print(f"\n📊 OFFLINE SQLITE EVALUATION (Semantic 'Vibe' Match for {len(gru_sims)} Users)")
print(f" ├─ GRU Neural Network Avg Similarity: {np.mean(gru_sims):.4f}")
print(f" └─ Heuristic Engine Avg Similarity:  {np.mean(heur_sims):.4f}")


Fetching history and recommendations directly from SQLite...
Searching for the downloaded YAMDA dataset...
✅ Found local embeddings file at: /root/.cache/kagglehub/datasets/ploshkin/yambda/versions/1/embeddings.parquet
Fetching vectors for fast comparison...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculating Average Semantic Similarity Matrix...

📊 OFFLINE SQLITE EVALUATION (Semantic 'Vibe' Match for 425 Users)
 ├─ GRU Neural Network Avg Similarity: 0.1541
 └─ Heuristic Engine Avg Similarity:  0.1655
